# CSCI 4150 · Meeting 7: Meaning as Geometry

**Meeting 7 · Fri Sep 18, 2026 · 10:00–11:40 AM**

Central question: **If meaning becomes distance, what does “close” mean?**

Run the cells in order. Your goal is to produce an **embedding-neighbor experiment** for the project dossier: a baseline query, one controlled change, one useful neighbor, and one related-but-wrong neighbor.

## Activity checklist

By the end, save or copy:

1. The exact baseline query.
2. The top 3 retrieved documents and similarity scores.
3. One controlled change and the new result.
4. One useful neighbor and one related-but-wrong neighbor.
5. A short note explaining what the system should not claim from similarity alone.

In [ ]:
# If you are in Colab and sentence-transformers is missing, run this cell.
# It is safe to skip if the import already works.
%pip install -q sentence-transformers scikit-learn pandas

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

pd.set_option('display.max_colwidth', 120)

## 1. Define the corpus

Each string below is one searchable document. In a real system, these could be policies, catalog entries, tickets, product descriptions, emails, or reports.

In [ ]:
documents = [
    'Students may request an extension by contacting the instructor before the deadline.',
    'Machine Learning requires Data Structures and Linear Algebra.',
    'The AI ethics seminar examines bias, privacy, labor, and governance.',
    'The lab safety manual requires goggles when working with corrosive chemicals.',
    'The course catalog from 2023 is archived and may not reflect current requirements.',
    'Office hours are available after class and by appointment.'
]

for i, doc in enumerate(documents, start=1):
    print(f'D{i}: {doc}')

D1: Students may request an extension by contacting the instructor before the deadline.
D2: Machine Learning requires Data Structures and Linear Algebra.
D3: The AI ethics seminar examines bias, privacy, labor, and governance.
D4: The lab safety manual requires goggles when working with corrosive chemicals.
D5: The course catalog from 2023 is archived and may not reflect current requirements.
D6: Office hours are available after class and by appointment.


## 2. Encode the documents

The embedding model turns each document into a numerical vector. We compare vectors, not raw sentences.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_vectors = model.encode(documents)

print('Number of documents:', len(documents))
print('Vector shape:', doc_vectors.shape)
print('First five numbers in D1 vector:', doc_vectors[0][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of documents: 6
Vector shape: (6, 384)
First five numbers in D1 vector: [-0.01688363  0.00146032  0.00021387 -0.0186613  -0.01001501]


## 3. Search function

Cosine similarity compares the direction of the query vector with each document vector. Higher scores usually mean closer semantic relatedness, but not guaranteed correctness.

In [ ]:
def search(query, k=3, docs=None, vectors=None):
    if docs is None:
        docs = documents
    if vectors is None:
        vectors = doc_vectors

    query_vector = model.encode([query])
    scores = cosine_similarity(query_vector, vectors)[0]
    ranked_indices = scores.argsort()[::-1][:k]

    rows = []
    for rank, idx in enumerate(ranked_indices, start=1):
        rows.append({
            'rank': rank,
            'document_id': f'D{idx + 1}',
            'score': round(float(scores[idx]), 4),
            'document': docs[idx],
        })
    return pd.DataFrame(rows)

## 4. Baseline query

Record the exact query, top 3 documents, and scores. Then decide which neighbor is useful and which is related-but-wrong.

In [ ]:
baseline_query = 'Can I take machine learning next semester?'
baseline_results = search(baseline_query, k=3)
baseline_results

,rank,document_id,score,document
0,1,D2,0.4437,Machine Learning requires Data Structures and Linear Algebra.
1,2,D5,0.3541,The course catalog from 2023 is archived and may not reflect current requirements.
2,3,D1,0.3394,Students may request an extension by contacting the instructor before the deadline.


### Baseline reflection

Answer in your own words after looking at the table:

- Which result is useful? Why?
- Which result is related-but-wrong or limited? Why?
- What information is missing before the system could answer the user’s actual question?

## 5. Controlled change A: reword the query

Keep the corpus fixed. Change only the query wording and compare the results.

In [ ]:
reworded_query = 'What are the prerequisites for Machine Learning?'
reworded_results = search(reworded_query, k=3)
reworded_results

,rank,document_id,score,document
0,1,D2,0.6461,Machine Learning requires Data Structures and Linear Algebra.
1,2,D3,0.1890,"The AI ethics seminar examines bias, privacy, labor, and governance."
2,3,D5,0.1671,The course catalog from 2023 is archived and may not reflect current requirements.


## 6. Controlled change B: add one document

Keep the baseline query fixed. Add one document and compare the new ranking.

In [ ]:
added_document = 'The 2026 catalog lists Machine Learning as requiring Data Structures, Linear Algebra, and instructor permission.'
expanded_documents = documents + [added_document]
expanded_vectors = model.encode(expanded_documents)

expanded_results = search(baseline_query, k=4, docs=expanded_documents, vectors=expanded_vectors)
expanded_results

,rank,document_id,score,document
0,1,D7,0.5246,"The 2026 catalog lists Machine Learning as requiring Data Structures, Linear Algebra, and instructor permission."
1,2,D2,0.4437,Machine Learning requires Data Structures and Linear Algebra.
2,3,D5,0.3541,The course catalog from 2023 is archived and may not reflect current requirements.
3,4,D1,0.3394,Students may request an extension by contacting the instructor before the deadline.


## 7. Evidence table for your dossier

Fill the `interpretation` fields after you inspect the output tables. This table is the artifact you can paste into the project dossier.

In [ ]:
evidence = pd.DataFrame([
    {
        'run': 'A: baseline',
        'query_or_change': baseline_query,
        'top_result': baseline_results.loc[0, 'document'],
        'top_score': baseline_results.loc[0, 'score'],
        'interpretation': 'Fill in: what did this ranking reveal?'
    },
    {
        'run': 'B: reworded query',
        'query_or_change': reworded_query,
        'top_result': reworded_results.loc[0, 'document'],
        'top_score': reworded_results.loc[0, 'score'],
        'interpretation': 'Fill in: what changed when the query changed?'
    },
    {
        'run': 'C: added document',
        'query_or_change': 'Added one 2026 catalog document; used baseline query',
        'top_result': expanded_results.loc[0, 'document'],
        'top_score': expanded_results.loc[0, 'score'],
        'interpretation': 'Fill in: what changed when the corpus changed?'
    },
])

evidence

,run,query_or_change,top_result,top_score,interpretation
0,A: baseline,Can I take machine learning next semester?,Machine Learning requires Data Structures and Linear Algebra.,0.4437,Fill in: what did this ranking reveal?
1,B: reworded query,What are the prerequisites for Machine Learning?,Machine Learning requires Data Structures and Linear Algebra.,0.6461,Fill in: what changed when the query changed?
2,C: added document,Added one 2026 catalog document; used baseline query,"The 2026 catalog lists Machine Learning as requiring Data Structures, Linear Algebra, and instructor permission.",0.5246,Fill in: what changed when the corpus changed?


## 8. Final reflection prompts

Write short answers before you leave:

1. What component did you inspect or modify today?
2. What assumption became visible?
3. What failure or limitation did you observe?
4. What object might your project embed?
5. What should similarity mean in your project, and what should it not be allowed to imply?